[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/47_moe_core_solution.ipynb)

# Solution: MoE Core (Simplified Dense Routing)

Reference solution.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✅ SOLUTION

class SimpleMoECore(nn.Module):
    def __init__(self, d_model, num_experts):
        super().__init__()
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([
            nn.Linear(d_model, d_model) for _ in range(num_experts)
        ])

    def forward(self, x):
        logits = self.router(x)
        weights = torch.softmax(logits, dim=-1)
        expert_out = torch.stack([expert(x) for expert in self.experts], dim=2)
        return (weights.unsqueeze(-1) * expert_out).sum(dim=2)


In [ ]:
# Demo
moe = SimpleMoECore(d_model=16, num_experts=4)
x = torch.randn(2, 6, 16)
print('Output shape:', moe(x).shape)

In [ ]:
from torch_judge import check
check('moe_core')